# Customer Lifetime Value & Customer Scoring

คำนวณ CLV และสร้าง Scoring Model สำหรับจัดลำดับลูกค้า

## 1. โหลดข้อมูล

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
transactions = pd.read_csv('../data/sample/transactions.csv', parse_dates=['transaction_date'])
customers    = pd.read_csv('../data/sample/customers.csv', parse_dates=['registration_date'])
products     = pd.read_csv('../data/sample/products.csv')

## 2. RFM Analysis

In [ ]:
snapshot_date = transactions['transaction_date'].max() + pd.Timedelta(days=1)

rfm = transactions.groupby('customer_id').agg(
    recency=('transaction_date', lambda x: (snapshot_date - x.max()).days),
    frequency=('transaction_id', 'count'),
    monetary=('amount', 'sum')
).reset_index()

rfm['R_quartile'] = pd.qcut(rfm['recency'], 4, labels=['1','2','3','4'])
rfm['F_quartile'] = pd.qcut(rfm['frequency'].rank(method='first'), 4, labels=['4','3','2','1'])
rfm['M_quartile'] = pd.qcut(rfm['monetary'], 4, labels=['4','3','2','1'])
rfm['RFM_score'] = rfm['R_quartile'].astype(str) + rfm['F_quartile'].astype(str) + rfm['M_quartile'].astype(str)
rfm.head()

In [ ]:
rfm['RFM_score'] = rfm[['R_quartile','F_quartile','M_quartile']].astype(int).sum(axis=1)
rfm.groupby('RFM_score').agg(count=('customer_id','count'), avg_monetary=('monetary','mean'))

## 3. CLV Calculation (Historical + Predictive)

In [ ]:
# Historical CLV — total spend per customer
clv_hist = transactions.groupby('customer_id')['amount'].sum().reset_index()
clv_hist.columns = ['customer_id', 'historical_clv']

# Predictive CLV — Simple BG/NBD-inspired (average spend × frequency × retention proxy)
avg_order = transactions.groupby('customer_id')['amount'].mean().reset_index()
avg_order.columns = ['customer_id', 'avg_order_value']

clv = clv_hist.merge(avg_order, on='customer_id')
clv['predicted_clv_12mo'] = clv['avg_order_value'] * rfm['frequency'] * 0.6  # simplified retention
clv.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1,2,1)
sns.histplot(clv['historical_clv'], bins=40, kde=True)
plt.title('Historical CLV Distribution')
plt.subplot(1,2,2)
sns.histplot(clv['predicted_clv_12mo'], bins=40, kde=True, color='orange')
plt.title('Predicted CLV (next 12mo)')
plt.tight_layout()
plt.savefig('../data/sample/clv_distributions.png', dpi=100)
plt.show()

## 4. Customer Scoring Model

In [ ]:
# Normalize features to 0-100
from sklearn.preprocessing import MinMaxScaler

scoring = rfm.merge(customers[['customer_id','age','income']], on='customer_id')
scaler = MinMaxScaler(feature_range=(0, 100))
scoring['score_recency']    = 100 - scaler.fit_transform(scoring[['recency']]) * 100  # lower is better
scoring['score_frequency']   = scaler.fit_transform(scoring[['frequency']]) * 100
scoring['score_monetary']    = scaler.fit_transform(scoring[['monetary']]) * 100
scoring['score_income']      = scaler.fit_transform(scoring[['income']]) * 100

# Composite score (custom weights: R=30%, F=25%, M=30%, Income=15%)
scoring['composite_score'] = (
    scoring['score_recency'] * 0.30 +
    scoring['score_frequency'] * 0.25 +
    scoring['score_monetary'] * 0.30 +
    scoring['score_income'] * 0.15
)
scoring['tier'] = pd.qcut(scoring['composite_score'], 4, labels=['Bronze','Silver','Gold','Platinum'])
scoring.sort_values('composite_score', ascending=False).head(10)

In [ ]:
tier_order = ['Bronze','Silver','Gold','Platinum']
plt.figure(figsize=(10, 4))
sns.boxplot(data=scoring, x='tier', y='monetary', order=tier_order, palette='viridis')
plt.title('Spending Distribution by Customer Tier')
plt.savefig('../data/sample/tier_spending.png', dpi=100)
plt.show()

## 5. สรุป

✅ คำนวณ RFM, CLV (historical + predictive) และ Customer Scoring สำเร็จ — จัดลำดับลูกค้าเป็น 4 tiers พร้อมนำไปใช้ต่อยอด